# Setup

In [ ]:
%run common.py
import sys
sys.path.append("../../legal-data-clustering/")
%run '../../legal-data-clustering/legal_data_clustering/utils/graph_api.py'
import multiprocessing

In [ ]:
def abs_to_rel(abs_array):
    rel_array = abs_array / abs_array[0]
    return rel_array

In [ ]:
us_crossreference_path = '../../legal-networks-data/us/4_crossreference_graph/seqitems'
de_crossreference_path = '../../legal-networks-data/de/4_crossreference_graph/seqitems'

In [ ]:
de_graph_files = sorted(list_dir(de_crossreference_path, 'gpickle.gz'))
de_graphs = [nx.read_gpickle(f'{de_crossreference_path}/{gf}') for gf in de_graph_files]

In [ ]:
us_graph_files = sorted(list_dir(us_crossreference_path, 'gpickle.gz'))
us_graphs = [nx.read_gpickle(f'{us_crossreference_path}/{gf}') for gf in us_graph_files]

In [ ]:
years = list(range(1994,1994+min(len(de_graphs),len(us_graphs)))) # usually our xs

# Generate law Graphs (Gesetze/Bücher/Chapter)

In [ ]:
def remove_empty(G: nx.Graph):
    return nx.subgraph(G, [n for n in G.nodes if G.nodes[n]['chars_n'] > 0]).copy()

us_law_graphs = [
    remove_empty(nx.read_gpickle(f'../../legal-networks-data/us/10_preprocessed_graph/{y}_0-0_1-0_-1.gpickle.gz'))
    for y in years
]

de_law_graphs = [
    remove_empty(nx.read_gpickle(f'../../legal-networks-data/de/10_preprocessed_graph/{y}-01-01_0-0_1-0_-1.gpickle.gz'))
    for y in years
]

# Anzahl

In [ ]:
def get_references(G: nx.MultiDiGraph):
    return [ 
        (u, v, k, edge_type)
        for u, v, k, edge_type 
        in G.edges(data='edge_type', keys=True)
        if edge_type == "reference"
    ]

In [ ]:
de_crossreferences_abs = np.array([len(get_references(G)) for G in de_graphs])
us_crossreferences_abs = np.array([len(get_references(G)) for G in us_graphs])
de_crossreferences_rel = abs_to_rel(de_crossreferences_abs)
us_crossreferences_rel = abs_to_rel(us_crossreferences_abs)

de_law_graph_crossreferences_abs = np.array([len(get_references(G)) for G in de_law_graphs])
us_law_graph_crossreferences_abs = np.array([len(get_references(G)) for G in us_law_graphs])
de_law_graph_crossreferences_rel = abs_to_rel(de_law_graph_crossreferences_abs)
us_law_graph_crossreferences_rel = abs_to_rel(us_law_graph_crossreferences_abs)

In [ ]:
df_crossreferences = pd.DataFrame({
    'USA': us_crossreferences_rel,
    'USA (absolut)': us_crossreferences_abs,
    'Deutschland': de_crossreferences_rel,
    'Deutschland (absolut)': de_crossreferences_abs,
    'USA Chapter': us_law_graph_crossreferences_rel,
    'USA Chapter (absolut)': us_law_graph_crossreferences_abs,
    'Deutschland Gesetze/Bücher': de_law_graph_crossreferences_rel,
    'Deutschland Gesetze/Bücher (absolut)': de_law_graph_crossreferences_abs,
    'Jahr': years
})

In [ ]:
diss_data('makro_crossrefs_us_erstes_jahr', de_num_format(f"{df_crossreferences['USA (absolut)'].iloc[0]:,}"))
diss_data('makro_crossrefs_us_letztes_jahr', de_num_format(f"{df_crossreferences['USA (absolut)'].iloc[-1]:,}"))
diss_data('makro_crossrefs_us_wachstumsfaktor', de_num_format(
    f"{df_crossreferences['USA (absolut)'].iloc[-1]/df_crossreferences['USA (absolut)'].iloc[0]:,.2f}"
))
diss_data('makro_crossrefs_de_erstes_jahr', de_num_format(f"{df_crossreferences['Deutschland (absolut)'].iloc[0]:,}"))
diss_data('makro_crossrefs_de_letztes_jahr', de_num_format(f"{df_crossreferences['Deutschland (absolut)'].iloc[-1]:,}"))
diss_data('makro_crossrefs_de_wachstumsfaktor', de_num_format(
    f"{df_crossreferences['Deutschland (absolut)'].iloc[-1]/df_crossreferences['Deutschland (absolut)'].iloc[0]:,.2f}"
))

diss_data('makro_crossrefs_buecher_de_erstes_jahr', de_num_format(f"{df_crossreferences['Deutschland Gesetze/Bücher (absolut)'].iloc[0]:,}"))
diss_data('makro_crossrefs_chapter_us_erstes_jahr', de_num_format(f"{df_crossreferences['USA Chapter (absolut)'].iloc[0]:,}"))

diss_data('makro_crossrefs_buecher_de_wachstumsfaktor', de_num_format(
    f"{df_crossreferences['Deutschland Gesetze/Bücher (absolut)'].iloc[-1]/df_crossreferences['Deutschland Gesetze/Bücher (absolut)'].iloc[0]:,.2f}"
))
diss_data('makro_crossrefs_chapter_us_wachstumsfaktor', de_num_format(
    f"{df_crossreferences['USA Chapter (absolut)'].iloc[-1]/df_crossreferences['USA Chapter (absolut)'].iloc[0]:,.2f}"
))

In [ ]:
df = df_crossreferences.copy()

assert list(df.columns) == [
    'USA', 
    'USA (absolut)', 
    'Deutschland', 
    'Deutschland (absolut)',
    'USA Chapter', 
    'USA Chapter (absolut)', 
    'Deutschland Gesetze/Bücher',
    'Deutschland Gesetze/Bücher (absolut)', 
    'Jahr'
]

df.columns = [
    'USA (Sections)', 
    'USA (absolut)', 
    'Deutschland (§/Artikel)', 
    'Deutschland (absolut)',
    'USA (Chapters)', 
    'USA Chapter (absolut)', 
    'Deutschland (Gesetze/Bücher)',
    'Deutschland Gesetze/Bücher (absolut)', 
    'Jahr'
]

In [ ]:
chart1 = alt.Chart(df).transform_fold(
    ['USA (Chapters)', 'USA (Sections)', 'Deutschland (Gesetze/Bücher)', 'Deutschland (§/Artikel)'],
).mark_line().encode(
    alt.X('Jahr:O'),
    alt.Y('value:Q', scale=alt.Scale(zero=False, clamp=True), title=f'Querverweise relativ zu {years[0]}'),
    alt.Color(
        'key:N', 
        legend=None,
        scale=alt.Scale(scheme='tableau20')
    ),
    alt.Tooltip(df.columns.to_list()),
)
chart2 = chart1.mark_point(filled=True, opacity=1).encode(
    alt.Color(
        'key:N', 
        legend=alt.Legend(
            title='Land (Auflösung)', legendX=10, legendY=2, orient='none', fillColor="#fff", padding=10
        ),
        scale=alt.Scale(scheme='tableau20')
    ),
    alt.Shape('key:N', scale=alt.Scale(range=['circle','square','triangle-up','triangle-down'])),
)
chart = (chart1 + chart2).resolve_scale(strokeDash="independent", shape='independent', color='independent')
save_chart(chart, 'makro_crossrefs_relative')

In [ ]:
chart1_grayscale = chart1.encode(
    alt.Color(
        'key:N', 
        legend=None,
        scale=alt.Scale(range=['#787878', '#BBB', '#000', '#9a9a9a']),
    ),
)
chart2_grayscale = chart2.encode(
    alt.Color(
        'key:N', 
        legend=alt.Legend(title='Land (Auflösung)', legendX=10, legendY=2, orient='none', fillColor="#fff", padding=10),
        scale=alt.Scale(range=['#787878', '#BBB', '#000', '#9a9a9a']),
    ),
)
chart_grayscale = (chart1_grayscale + chart2_grayscale).resolve_scale(strokeDash="independent", shape='independent', color='independent')
save_chart(chart_grayscale, 'makro_crossrefs_relative_graycolor')

## Welche Gesetze sind für das Wachstum in DE ab 2015 verantwortlich?

In [ ]:
G1 = de_graphs[2015-1994]
G2 = de_graphs[2018-1994]

H1 = filter_edges(G1, 'edge_type', 'containment')
H2 = filter_edges(G2, 'edge_type', 'containment')

outgoing_ref_counts1 = Counter([u.split('_')[1] for u, v, k in H1.edges])
outgoing_ref_counts2 = Counter([u.split('_')[1] for u, v, k in H2.edges])

keys = sorted(set(outgoing_ref_counts1) | set(outgoing_ref_counts2))

cross_ref_df = pd.DataFrame([
    [
        outgoing_ref_counts1[k],
        outgoing_ref_counts2[k],
    ]
    for k in keys
], index=keys)

In [ ]:
cross_ref_df['diff'] = cross_ref_df[1] - cross_ref_df[0]

In [ ]:
cross_ref_df.sort_values('diff', ascending=True)[-20:]

## Vergleich mit Tokens

In [ ]:
df_tokens = pd.read_pickle(f'{data_pickles_path}/makro_tokens.pickle')
assert df_tokens['Jahr'].to_list() == df_crossreferences['Jahr'].to_list()

In [ ]:
df_tokens_crossreferences = pd.DataFrame({
    'Deutschland': (
        df_tokens['Deutschland (absolut)'] / 
        df_crossreferences['Deutschland (absolut)']
    ),
    'USA': (
        df_tokens['USA (absolut)'] / 
        df_crossreferences['USA (absolut)']
    ),
    'Jahr': years
})

In [ ]:
df_tokens_crossreferences

In [ ]:
diss_data('makro_crossrefs_tokens_us_min', de_num_format(f"{df_tokens_crossreferences['USA'].min():,.2f}"))
diss_data('makro_crossrefs_tokens_us_max', de_num_format(f"{df_tokens_crossreferences['USA'].max():,.2f}"))
diss_data('makro_crossrefs_tokens_de_erstes_jahr', de_num_format(f"{df_tokens_crossreferences['Deutschland'].iloc[0]:,.2f}"))
diss_data('makro_crossrefs_tokens_de_letztes_jahr', de_num_format(f"{df_tokens_crossreferences['Deutschland'].iloc[-1]:,.2f}"))
diss_data('makro_crossrefs_tokens_de_prozent', de_num_format(
    f"{100-(df_tokens_crossreferences['Deutschland'].iloc[-1]/df_tokens_crossreferences['Deutschland'].iloc[0]*100):,.2f}"
))

In [ ]:
chart_de = alt.Chart(
    df_tokens_crossreferences, 
).mark_line(
    point=alt.OverlayMarkDef(color='#1f77b4'),
    color='#1f77b4',
).encode(
    alt.X('Jahr:O',
         axis=alt.Axis(ticks=False, labels=False, title=None)
    ),
    alt.Y('Deutschland:Q', 
          scale=alt.Scale(zero=False, domain=[53, 59]),
          axis=alt.Axis(tickCount=7, tickMinStep=2,titlePadding=9),
          title=['Tokens /', ' Gesetzeszitate'],
         ),
    ).properties(height=70)
chart_de

In [ ]:
chart_us = alt.Chart(
    df_tokens_crossreferences,
).mark_line(
    point=alt.OverlayMarkDef(color='#ff7f0e', shape="triangle-up"),
    color='#ff7f0e'
).encode(
    alt.X('Jahr:O',
          axis=alt.Axis(ticks=False, labels=False, title=None)
         ),
    alt.Y('USA:Q', 
          scale=alt.Scale(zero=False, domain=[189, 195]),
          axis=alt.Axis(tickCount=6, tickMinStep=2),
          title=['Tokens /', ' Gesetzeszitate'],
         ),
).properties(height=70)
chart_us

In [ ]:
chart_de_graycolor = chart_de.mark_line(point=alt.OverlayMarkDef(color="#888"), color="#888")
chart_de_graycolor

In [ ]:
chart_us_graycolor = chart_us.mark_line(point=alt.OverlayMarkDef(color="#000", shape="triangle-up"), color="#000")
chart_us_graycolor

### Vergleich mit indexierten Elementen

In [ ]:
df_seqitems = pd.read_pickle(f'{data_pickles_path}/makro_seqitems.pickle')
assert df_seqitems['Jahr'].to_list() == df_crossreferences['Jahr'].to_list()

In [ ]:
df_crossreferences_seqitems = pd.DataFrame({
    'Deutschland': (
        df_crossreferences['Deutschland (absolut)'] /
        df_seqitems['Deutschland (absolut)']
    ),
    'USA': (
        df_crossreferences['USA (absolut)'] /
        df_seqitems['USA (absolut)']
    ),
    'Jahr': years
})

In [ ]:
diss_data('makro_crossrefs_seqitems_us_erstes_jahr', de_num_format(f"{df_crossreferences_seqitems['USA'].iloc[0]:,.2f}"))
diss_data('makro_crossrefs_seqitems_us_letztes_jahr', de_num_format(f"{df_crossreferences_seqitems['USA'].iloc[-1]:,.2f}"))
diss_data('makro_crossrefs_seqitems_us_wachstumsfaktor', de_num_format(
    f"{df_crossreferences_seqitems['USA'].iloc[-1]/df_crossreferences_seqitems['USA'].iloc[0]:,.2f}"
))
diss_data('makro_crossrefs_seqitems_de_erstes_jahr', de_num_format(f"{df_crossreferences_seqitems['Deutschland'].iloc[0]:,.2f}"))
diss_data('makro_crossrefs_seqitems_de_letztes_jahr', de_num_format(f"{df_crossreferences_seqitems['Deutschland'].iloc[-1]:,.2f}"))
diss_data('makro_crossrefs_seqitems_de_wachstumsfaktor', de_num_format(
    f"{df_crossreferences_seqitems['Deutschland'].iloc[-1]/df_crossreferences_seqitems['Deutschland'].iloc[0]:,.2f}"
))

In [ ]:
chart_seqitems1 = alt.Chart(
    df_crossreferences_seqitems, 
).transform_fold(
    ['USA', 'Deutschland'],
).mark_line().encode(
    alt.X('Jahr:O'),
    alt.Y('value:Q', 
        scale=alt.Scale(zero=False),
        axis=alt.Axis(tickCount=10, tickMinStep=.2),
        title=['Gesetzeszitate /', ' Indexierte Elemente'],
    ),
    alt.Color('key:N', legend=None),
).properties(height=160)
chart_seqitems2 = chart_seqitems1.mark_point(filled=True, opacity=1).encode(
    alt.Color('key:N', legend=alt.Legend(title='Land', legendX=10, legendY=2, orient='none', fillColor="#fff", padding=10),),
    alt.Shape('key:N', scale=alt.Scale(range=['circle','triangle-up']), legend=None),
)
chart_seqitems = (chart_seqitems1 + chart_seqitems2).resolve_scale(shape='independent', color='independent')
chart_seqitems

In [ ]:
chart_seqitems1_graycolor = chart_seqitems1.encode(
    alt.Color('key:N', legend=None, scale=alt.Scale(range=["#888", "#000"])),
)
chart_seqitems2_graycolor = chart_seqitems2.encode(
    alt.Color('key:N', legend=alt.Legend(title='Land', legendX=10, legendY=2, orient='none', fillColor="#fff", padding=10), scale=alt.Scale(range=["#888", "#000"])),
)
chart_seqitems_graycolor = (chart_seqitems1_graycolor + chart_seqitems2_graycolor).resolve_scale(shape='independent', color='independent')
chart_seqitems_graycolor

### Vergleich mit ungeordneten und geordneten Elementen

In [ ]:
df_items = pd.read_pickle(f'{data_pickles_path}/makro_items.pickle')
assert df_items['Jahr'].to_list() == df_crossreferences['Jahr'].to_list()

In [ ]:
df_crossreferences_items = pd.DataFrame({
    'Deutschland': (
        df_crossreferences['Deutschland (absolut)'] /
        df_items['Deutschland (absolut)']
    ),
    'USA': (
        df_crossreferences['USA (absolut)'] /
        df_items['USA (absolut)']
    ),
    'Jahr': years
})

In [ ]:
diss_data('makro_crossrefs_items_us_erstes_jahr', de_num_format(f"{df_crossreferences_items['USA'].iloc[0]:,.2f}"))
diss_data('makro_crossrefs_items_us_letztes_jahr', de_num_format(f"{df_crossreferences_items['USA'].iloc[-1]:,.2f}"))
diss_data('makro_crossrefs_items_us_wachstumsfaktor', de_num_format(
    f"{df_crossreferences_items['USA'].iloc[-1]/df_crossreferences_items['USA'].iloc[0]:,.2f}"
))
diss_data('makro_crossrefs_items_de_erstes_jahr', de_num_format(f"{df_crossreferences_items['Deutschland'].iloc[0]:,.2f}"))
diss_data('makro_crossrefs_items_de_letztes_jahr', de_num_format(f"{df_crossreferences_items['Deutschland'].iloc[-1]:,.2f}"))
diss_data('makro_crossrefs_items_de_wachstumsfaktor', de_num_format(
    f"{df_crossreferences_items['Deutschland'].iloc[-1]/df_crossreferences_items['Deutschland'].iloc[0]:,.2f}"
))

In [ ]:
chart_items = alt.Chart(
    df_crossreferences_items, 
).transform_fold(
    ['USA', 'Deutschland'],
).mark_line(
   point=True,
).encode(
    alt.X('Jahr:O'),
    alt.Y('value:Q', 
          scale=alt.Scale(
              zero=False, 
#               domain=[52, 62]
          ),
          axis=alt.Axis(tickCount=10, tickMinStep=.05),
          title=['Gesetzeszitate /', 'ungeordnete und geordnete Elemente'],
         ),
    alt.Color('key:N', legend=alt.Legend(
        title='Land',
    )),
    ).properties(height=160)
chart_items

### Combine Charts

In [ ]:
chart = alt.vconcat(
    chart_us, chart_de, chart_seqitems
).configure_concat(spacing=7)
save_chart(chart, 'makro_crossrefs_rel_tokens_seqitems')

In [ ]:
chart_graycolor = alt.vconcat(
    chart_us_graycolor, chart_de_graycolor, chart_seqitems_graycolor
).configure_concat(spacing=7)
save_chart(chart_graycolor, 'makro_crossrefs_rel_tokens_seqitems_graycolor')

# Verweis in anderes Gesetz, Title, Chapter

In [ ]:
def get_references_other_law_title(G: nx.MultiDiGraph):
    return [ 
        (u, v, k, data)
        for u, v, k, data 
        in G.edges(data=True, keys=True)
        if data['edge_type'] == "reference" and u.split('_')[0] != v.split('_')[0]
    ]


def get_references_other_law_title_graph(G: nx.MultiDiGraph):
    sG = nx.MultiDiGraph()
    sG.add_nodes_from(G.nodes(data=True))
    sG.add_edges_from(get_references_other_law_title(G))
    return sG


def get_all_reference_edges_graph(G):
    return induced_subgraph(G, 'edge', 'edge_type', ['reference'])


def get_references_other_chapter_len(G: nx.MultiDiGraph):
    return len(get_references_other_chapter(G))

In [ ]:
de_crossreferences_gesetz_abs = np.array([len(get_references_other_law_title(G)) for G in de_graphs])
us_crossreferences_title_abs = np.array([len(get_references_other_law_title(G)) for G in us_graphs])
de_crossreferences_gesetz_rel = abs_to_rel(de_crossreferences_gesetz_abs)
us_crossreferences_title_rel = abs_to_rel(us_crossreferences_title_abs)

us_crossreferences_chapter_abs = np.array([len(G.edges) for G in us_law_graphs])
us_crossreferences_chapter_rel = abs_to_rel(us_crossreferences_chapter_abs)

In [ ]:
df_crossreferences_other_items =  pd.DataFrame({
        'USA Title': us_crossreferences_title_rel,
        'USA Title (absolut)': us_crossreferences_title_abs,
        'USA Chapter': us_crossreferences_chapter_rel,
        'USA Chapter (absolut)': us_crossreferences_chapter_abs,
        'Deutschland Gesetz': de_crossreferences_gesetz_rel,
        'Deutschland Gesetz (absolut)': de_crossreferences_gesetz_abs,
        'Jahr': years,
    })

In [ ]:
df_crossreferences_other_items

In [ ]:
chart = alt.Chart(df_crossreferences_other_items).transform_fold(
    ['Deutschland Gesetz', 'USA Title', 'USA Chapter'],
).mark_line(point=True).encode(
    alt.X('Jahr:O'),
    alt.Y('value:Q', scale=alt.Scale(zero=False), title=f'Querverweise relativ zu {years[0]}'),
    alt.Color(
        'key:N', 
        legend=alt.Legend(title=None, legendX=10, legendY=2, orient='none', fillColor="#fff", padding=10),
        sort=['Deutschland Gesetz', 'USA Title', 'USA Chapter']
    ),
    alt.Tooltip(df_crossreferences_other_items.columns.to_list()),
)
chart

In [ ]:
df_crossreferences_other_items_ratios = pd.DataFrame({
    "Deutschland": (
        df_crossreferences_other_items['Deutschland Gesetz (absolut)']/ 
        df_crossreferences['Deutschland (absolut)']
    ),
    "USA": (
        df_crossreferences_other_items['USA Title (absolut)']/ 
        df_crossreferences['USA (absolut)']
    ),
    "USA Chapter": (
        df_crossreferences_other_items['USA Chapter (absolut)']/ 
        df_crossreferences['USA (absolut)']
    ),
    "Jahr": years,
})

In [ ]:
df_crossreferences_other_items_ratios

diss_data('makro_crossrefs_other_chapter_us_erstes_jahr', de_num_format(f"{df_crossreferences_other_items_ratios['USA Chapter'].iloc[0]*100:,.2f}"))
diss_data('makro_crossrefs_other_chapter_us_letztes_jahr', de_num_format(f"{df_crossreferences_other_items_ratios['USA Chapter'].iloc[-1]*100:,.2f}"))
diss_data('makro_crossrefs_other_chapter_us_wachstumsfaktor', de_num_format(
    f"{df_crossreferences_other_items_ratios['USA Chapter'].iloc[-1]/df_crossreferences_other_items_ratios['USA Chapter'].iloc[0]:,.2f}"
))
diss_data('makro_crossrefs_other_items_de_erstes_jahr', de_num_format(f"{df_crossreferences_other_items_ratios['Deutschland'].iloc[0]*100:,.2f}"))
diss_data('makro_crossrefs_other_items_de_letztes_jahr', de_num_format(f"{df_crossreferences_other_items_ratios['Deutschland'].iloc[-1]*100:,.2f}"))
diss_data('makro_crossrefs_other_items_de_wachstumsfaktor', de_num_format(
    f"{df_crossreferences_other_items_ratios['Deutschland'].iloc[-1]/df_crossreferences_other_items_ratios['Deutschland'].iloc[0]:,.2f}"
))

In [ ]:
chart1 = alt.Chart(
    df_crossreferences_other_items_ratios,
    title=alt.TitleParams(
        'Anteil der Normzitate zwischen', 
        orient='left',
        fontSize=11,
        offset=2
    ),
).transform_fold(
    ['Deutschland', 'USA', 'USA Chapter'],
).mark_line().encode(
    alt.X('Jahr:O'),
    alt.Y('value:Q', scale=alt.Scale(zero=False, clamp=True), 
          title='Gesetzen/Titles/Chapters'
#           title="Anteil der Querverweise zwischen"
         ),
    alt.Color(
        'key:N', 
        legend=None,
        #sort=['Deutschland (Gesetze)', 'USA (Titles)', '(Chapters)'],
    ),
    alt.Tooltip(df_crossreferences_other_items_ratios.columns.to_list()),
)
chart2 = chart1.mark_point(filled=True, opacity=1).encode(
    alt.Shape('key:N', scale=alt.Scale(range=['circle','triangle-up','triangle-down'])),
    alt.Color(
        'key:N', 
        legend=alt.Legend(
            title=None, legendX=300, legendY=25, orient='none', fillColor="#fff", padding=10
        ),
        #sort=['Deutschland (Gesetze)', 'USA (Titles)', '(Chapters)'],
    ),
)
chart = (chart1 + chart2).resolve_scale(shape='independent', color='independent')
save_chart(chart, 'makro_crossrefs_gesetz_title_chapter')

In [ ]:
chart1_graycolor = chart1.encode(
    alt.Color(
        'key:N', 
        legend=None,
        scale=alt.Scale(range=['#888', '#000', '#AAA']),
    ),
)
chart2_graycolor = chart2.encode(
    alt.Color(
        'key:N', 
        legend=alt.Legend(title=None, legendX=300, legendY=25, orient='none', fillColor="#fff", padding=10),
        scale=alt.Scale(range=['#888', '#000', '#AAA']),
    ),
)
chart_graycolor = (chart1_graycolor + chart2_graycolor).resolve_scale(shape='independent', color='independent')
save_chart(chart_graycolor, 'makro_crossrefs_gesetz_title_chapter_graycolor')

## Gradverteilung

In [ ]:
def get_in_out_degrees_for_country(G, year, edge_filter_func):
    iG = edge_filter_func(G)
    out_degrees = list(dict(iG.out_degree()).values())
    in_degrees  = list(dict(iG.in_degree() ).values())
    out_degrees_df = binned_df(out_degrees, np.arange(-1, max(out_degrees)+1, 1), year)
    in_degrees_df = binned_df(in_degrees, np.arange(-1, max(in_degrees)+1, 1), year)
    return in_degrees_df, out_degrees_df

In [ ]:
def process_degree_df(dfs):
    in_degrees_df_de = pd.DataFrame()
    out_degrees_df_de = pd.DataFrame()

    in_degrees_df_us = pd.DataFrame()
    out_degrees_df_us = pd.DataFrame()

    for in_degrees_de_df, out_degrees_de_df, in_degrees_us_df, out_degrees_us_df in dfs:
        in_degrees_df_de  = in_degrees_df_de. append(in_degrees_de_df)
        out_degrees_df_de = out_degrees_df_de.append(out_degrees_de_df)
        in_degrees_df_us  = in_degrees_df_us. append(in_degrees_us_df)
        out_degrees_df_us = out_degrees_df_us.append(out_degrees_us_df)

    in_degrees_df_de['Land'] = 'Deutschland'
    in_degrees_df_de['Richtung'] = 'Eingehend'
    out_degrees_df_de['Land'] = 'Deutschland'
    out_degrees_df_de['Richtung'] = 'Ausgehend'
    in_degrees_df_us['Land'] = 'USA'
    in_degrees_df_us['Richtung'] = 'Eingehend'
    out_degrees_df_us['Land'] = 'USA'
    out_degrees_df_us['Richtung'] = 'Ausgehend'
    return pd.concat([in_degrees_df_de, out_degrees_df_de, in_degrees_df_us, out_degrees_df_us])

In [ ]:
def get_in_out_degrees(idx, edge_filter_func):
    '''
    edge_filter_func can be:
    - get_all_reference_edges_graph
    - get_references_other_law_title_graph
    - get_references_other_chapter_graph
    '''
    year = years[idx]
    G = working_de_graphs[idx]
    in_degrees_de_df, out_degrees_de_df = get_in_out_degrees_for_country(G, year, edge_filter_func)
    G = working_us_graphs[idx]
    in_degrees_us_df, out_degrees_us_df = get_in_out_degrees_for_country(G, year, edge_filter_func)
    return (in_degrees_de_df, out_degrees_de_df, in_degrees_us_df, out_degrees_us_df)


In [ ]:
def df_degree_bins_abs_to_rel(df):
    grouped_sums = df.groupby(['Jahr', 'Land']).sum().x
    df_degrees_rel = df.copy()
    df_degrees_rel.x = [row.x / grouped_sums[(row.Jahr, row.Land)] for idx, row in df.iterrows()]
    return df_degrees_rel

In [ ]:
working_de_graphs = de_graphs
working_us_graphs = us_graphs

with multiprocessing.Pool(5) as p:
#     dfs = p.map(get_in_out_degrees,   [(i, get_all_reference_edges_graph) for i in range(len(years))])
    dfs = p.starmap(get_in_out_degrees, [(i, get_all_reference_edges_graph) for i in [0, -1]]) # first and last year only
df_degrees_all = process_degree_df(dfs)
# None
# Achtung 0 hier ausgeblendet

In [ ]:
def degree_df_chart(df_degrees, y_label):
    df_degrees['merge'] = [f'{row.Land} {row.Jahr}' for idx, row in df_degrees.iterrows()]
    
    # Plot 1994 in front of 2019
    df_degrees = df_degrees.sort_values(['Land', 'Jahr'], ascending=False)
    
    df_degrees = df_degrees[
    (
        (df_degrees['Jahr'] == years[0]) |
        (df_degrees['Jahr'] == years[-1]) 
    ) & 
    (df_degrees.bin_max <= 25) & 
    (df_degrees.bin_max > 0)
]
    
    chart = alt.Chart(
        df_degrees
    ).mark_line(interpolate="step-after", size=1).encode(
        alt.X('bin_min',
            title="Verweise",
            scale=alt.Scale(nice=False),
        ),
        alt.Y('x:Q',
            title=y_label,
        ),
        alt.Color('Jahr:N', 
                  legend=alt.Legend(legendX=320, legendY=5, orient='none', fillColor="#fff", padding=10, title=None),
                  scale=alt.Scale(scheme='set1'),
                 ),
        alt.Column('Richtung:N', title=None),
        alt.Row('Land:N', title=None),
        alt.StrokeDash("Jahr:N", scale=alt.Scale(range=[[1, 0], [1, 1]])),
    ).properties(
        width=180, height=140, 
#         columns=2
    )
    return chart

In [ ]:
chart = degree_df_chart(df_degrees_all, "Indexierte Elemente")
save_chart(chart, 'makro_crossrefs_degree_dist_all')

In [ ]:
chart_graycolor = chart.encode(
    alt.Color('Jahr:N', 
        legend=alt.Legend(legendX=330, legendY=5, orient='none', fillColor="#fff", padding=10, title=None),
        scale=alt.Scale(range=['#000', '#666']),
    ),
)
save_chart(chart_graycolor, 'makro_crossrefs_degree_dist_all_graycolor')

### Power law?

In [ ]:
# Warning can be ignored https://github.com/jeffalstott/powerlaw/issues/25
for land in ['USA', 'Deutschland']:
    for richtung in ['Eingehend', 'Ausgehend']:
        for jahr in [1994, 2019]:
            data = df_degrees_all[
                (df_degrees_all.Land==land)&
                (df_degrees_all.Richtung==richtung)&
                (df_degrees_all.Jahr==jahr)
            ].x
            data = data[data>0]
            import powerlaw
            fit = powerlaw.Fit(list(data), discrete=True)
            print('---', land, richtung, jahr, '---')
            print(fit.distribution_compare('power_law', 'lognormal'))

### Relative Verteilung der Knotengrade

In [ ]:
df_degrees_all_rel = df_degree_bins_abs_to_rel(df_degrees_all)
chart = degree_df_chart(df_degrees_all_rel, 'Anteil indexierter Elemente')
chart
save_chart(chart, 'makro_crossrefs_degree_dist_all_rel')

In [ ]:
chart_graycolor = chart.encode(
    alt.Color('Jahr:N', 
        legend=alt.Legend(legendX=330, legendY=5, orient='none', fillColor="#fff", padding=10, title=None),
        scale=alt.Scale(range=['#000', '#666']),
    ),
)
save_chart(chart_graycolor, 'makro_crossrefs_degree_dist_all_rel_graycolor')

### Entwicklung

In [ ]:
def get_count_for_row_and_year(df, row, jahr):
    result_df = df[(df.Land==row.Land)&(df.Jahr==jahr)&(df.bin_max==row.bin_max)&(df.Richtung==row.Richtung)]
    assert len(result_df) < 2
    return result_df.x.iloc[0] if len(result_df) == 1 else 0

In [ ]:
df_degrees_all_rel_diff = df_degrees_all_rel[df_degrees_all_rel.Jahr==2019].copy()
df_degrees_all_rel_diff['x1994'] = [
    get_count_for_row_and_year(df_degrees_all_rel, row, 1994)
    for idx, row in df_degrees_all_rel_diff.iterrows()
]
df_degrees_all_rel_diff['diff'] =  (df_degrees_all_rel_diff.x - df_degrees_all_rel_diff.x1994) #/ df_degrees_all_rel_diff.x1994

plot_df = pd.pivot_table(
    df_degrees_all_rel_diff,
    values='diff',
    index=['bin_min', 'bin_max', 'Land'],
    columns='Richtung'
).reset_index()
len(plot_df)

In [ ]:
def plot_degrees_all_rel_diff(df_degrees_all_rel):

    df_degrees_all_rel_diff = df_degrees_all_rel[df_degrees_all_rel.Jahr==2019].copy()
    df_degrees_all_rel_diff['x1994'] = [
        get_count_for_row_and_year(df_degrees_all_rel, row, 1994)
        for idx, row in df_degrees_all_rel_diff.iterrows()
    ]
    df_degrees_all_rel_diff['diff'] =  (df_degrees_all_rel_diff.x - df_degrees_all_rel_diff.x1994) #/ df_degrees_all_rel_diff.x1994

    plot_df = pd.pivot_table(
        df_degrees_all_rel_diff,
        values='diff',
        index=['bin_min', 'bin_max', 'Land'],
        columns='Richtung'
    ).reset_index()
    assert len(plot_df)
    chart = alt.Chart(plot_df[
    #     (plot_df.bin_max > 0)&
        (plot_df.bin_max < 20+1)
    ]).transform_fold(['Ausgehend', 'Eingehend']).mark_line(point=True).encode(
        alt.X('bin_max', 
            title="Verweise",
            scale=alt.Scale(domain=(0, 20), clamp=True,),
        ),
        alt.Y('value:Q',
            title=["Veränderung des Anteils","der indexierten Elemente", f"von {years[0]} zu {years[-1]}"],
            axis=alt.Axis(tickCount=9),
            scale=alt.Scale(domain=(-0.005,0.005), clamp=True)
        ),
        alt.Color(
            'key:N',  
            scale=alt.Scale(scheme='set1'),
            title=None,
            legend=alt.Legend(legendX=340, legendY=5, orient='none', fillColor="#fff", padding=10, title=None),
        ),
        alt.Facet('Land:N')
    )
    return chart
chart = plot_degrees_all_rel_diff(df_degrees_all_rel)
save_chart(chart, 'makro_crossrefs_degree_dist_all_rel_diff')

### Nur entfernte Zitate aber weiterhin Analyse auf Level indexierter Elemente

Aktuell nicht genutzt

In [ ]:
# with multiprocessing.Pool(5) as p:
#     dfs = p.starmap(get_in_out_degrees, [(i, get_references_other_law_title_graph) for i in [0, -1]]) # first and last year only
# df_degrees = process_degree_df(dfs)
# chart = degree_df_chart(df_degrees, 'Indexierte Elemente')
# save_chart(chart, 'makro_crossrefs_degree_dist_gesetz_title')

# Achtung 0 hier ausgeblendet

In [ ]:
# df_degrees_rel = df_degree_bins_abs_to_rel(df_degrees)
# chart = degree_df_chart(df_degrees_rel, 'Indexierte Elemente')
# save_chart(chart, 'makro_crossrefs_degree_dist_gesetz_title_rel')

In [ ]:
# Warning unten

# with multiprocessing.Pool(5) as p:
# #     dfs = p.map(get_in_out_degrees,   [(i, get_references_other_chapter_graph) for i in range(len(years))])
#     dfs = p.starmap(get_in_out_degrees, [(i, get_references_other_chapter_graph) for i in [0, -1]]) # first and last year only
# df_degrees = process_degree_df(dfs)
# chart = degree_df_chart(df_degrees, 'Indexierte Elemente')
# chart

# Dieses Chart ist insbesondere für die US Daten interessant. 
# Die deutschen Daten sollten nicht verwendet werden, da nur 
# Zitate berücksichtigt werden, die zu einem anderen level 1 
# Zweig gehören. References auf Seqitems die ohne Items mit 
# dem Document verbunden sind, werden ignoriert. Nur USA nutzen!
# Achtung 0 hier ausgeblendet

### Zitate auf eben von Gesetzen bzw. Chapter

### Unterschied zu Anzahl DE Gesetze

In [ ]:
de_documents_n_abs = np.array([get_node_count_for_type(G, ['document']) for G in de_graphs])

# Compare
diss_data('makro_chapter_buch_de_erstes_jahr', de_num_format(f'{len(de_law_graphs[0].nodes):,}'))
diss_data('makro_chapter_buch_de_letztes_jahr', de_num_format(f'{len(de_law_graphs[-1].nodes):,}'))
diff_erstes_jahr = (len(de_law_graphs[0].nodes) - de_documents_n_abs[0]) / de_documents_n_abs[0] * 100
diff_letztes_jahr = (len(de_law_graphs[-1].nodes) - de_documents_n_abs[-1]) / de_documents_n_abs[-1] * 100
diss_data('makro_chapter_buch_de_erstes_jahr_diff', de_num_format(f'{diff_erstes_jahr:,.2f}'))
diss_data('makro_chapter_buch_de_letztes_jahr_diff', de_num_format(f'{diff_letztes_jahr:,.2f}'))


In [ ]:
working_de_graphs = de_law_graphs
working_us_graphs = us_law_graphs

In [ ]:
with multiprocessing.Pool(5) as p:
    dfs = p.starmap(get_in_out_degrees, [(i, get_all_reference_edges_graph) for i in [0, -1]]) # first and last year only
df_degrees_all = process_degree_df(dfs)
chart = degree_df_chart(df_degrees_all[df_degrees_all.bin_max < 21].copy(),
                        'Gesetze, Bücher oder Chapters'
                       )
save_chart(chart, 'makro_crossrefs_degree_dist_all_filtered')

In [ ]:
chart_graycolor = chart.encode(
    alt.Color('Jahr:N', 
        legend=alt.Legend(legendX=330, legendY=5, orient='none', fillColor="#fff", padding=10, title=None),
        scale=alt.Scale(range=['#000', '#666']),
    ),
)
save_chart(chart_graycolor, 'makro_crossrefs_degree_dist_all_filtered_graycolor')

In [ ]:
df_degrees_all_rel = df_degree_bins_abs_to_rel(df_degrees_all)
chart = degree_df_chart(
    df_degrees_all_rel[df_degrees_all_rel.bin_max < 21].copy(), 
    'Gesetze, Bücher oder Chapters'
)
chart
save_chart(chart, 'makro_crossrefs_degree_dist_all_rel_filtered')

In [ ]:
chart_graycolor = chart.encode(
    alt.Color('Jahr:N', 
        legend=alt.Legend(legendX=330, legendY=5, orient='none', fillColor="#fff", padding=10, title=None),
        scale=alt.Scale(range=['#000', '#666']),
    ),
)
save_chart(chart_graycolor, 'makro_crossrefs_degree_dist_all_rel_filtered_graycolor')

In [ ]:
# Power law?

for land in ['USA', 'Deutschland']:
    for richtung in ['Eingehend', 'Ausgehend']:
        for jahr in [1994, 2019]:
            data = df_degrees_all[
                (df_degrees_all.Land==land)&
                (df_degrees_all.Richtung==richtung)&
                (df_degrees_all.Jahr==jahr)
            ].x
            data = data[data>0]
            import powerlaw
            fit = powerlaw.Fit(data, discrete=True)
            print('---', land, richtung, jahr, '---')
            print(fit.distribution_compare('power_law', 'lognormal'))

In [ ]:
chart = plot_degrees_all_rel_diff(df_degrees_all_rel)
chart
# save_chart(chart, 'makro_crossrefs_degree_dist_all_rel_diff_filtered')